# 03 - Sparse Autoencoder Training & Semantic Grounding

## 1. Setup and Imports
Configure paths to seamlessly run on Colab or locally, and install required libraries.

In [1]:
import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    base_dir = '/content/drive/MyDrive/xai-project5'
else:
    base_dir = os.path.abspath(os.path.join('..', '..'))

feat_dir = os.path.join(base_dir, 'src', 'results', 'feature_extraction')
dict_dir = os.path.join(base_dir, 'src', 'results', '02_dictionary_creation')
save_dir = os.path.join(base_dir, 'src', 'results', '03_sae_training')
scripts_path = os.path.join(base_dir, 'src', 'scripts')
req_path = os.path.join(base_dir, 'requirements_ai_core.txt')

os.makedirs(save_dir, exist_ok=True)

if scripts_path not in sys.path:
    sys.path.append(scripts_path)

print(f"Setup complete.\nOutput directory: {save_dir}")


Setup complete.
Output directory: /home/emmanuelmessina00/Scrivania/xai-project5/src/results/03_sae_training


In [ ]:
!pip install -q -r "{req_path}" transformers pillow matplotlib seaborn


In [2]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoProcessor, AutoModel
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sae import SparseAutoencoder, sae_loss_function

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


## 2. Sparse Autoencoder Training

**Sparse Autoencoders (SAEs)** implement a form of *sparse dictionary learning*, aiming to learn a sparse decomposition of a signal into an overcomplete dictionary of atoms.

Given an embedding $v \in \mathbb{R}^d$, the SAE decomposes the vector into:
- **Activation vector**: $\phi(v) := \sigma(W_{enc}^{\top}(v - b))$
- **Reconstructed vector**: $\hat{v} := W_{dec}^{\top}\phi(v) + b$

The loss function combines a **reconstruction objective** with a **sparsity regularization**:
$$\mathcal{L}(v) = R(v) + \lambda S(v)$$
where $R(v) = \|v - \hat{v}\|_2^2$ (L2 Loss) and $S(v) = \|\phi(v)\|_1$ (L1 Loss).

In [3]:
embeddings_path = os.path.join(feat_dir, 'biomedclip_openi_with_reports.pt')
print(f"Loading visual dataset from: {embeddings_path}")
vision_embeddings = torch.load(embeddings_path, map_location=device)

BATCH_SIZE = 256
dataset = TensorDataset(vision_embeddings)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

INPUT_DIM = vision_embeddings.shape[1]
HIDDEN_DIM = 2048
LR = 1e-3
L1_LAMBDA = 1e-4
EPOCHS = 20

sae = SparseAutoencoder(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(device)
optimizer = optim.Adam(sae.parameters(), lr=LR)


Loading visual dataset from: /home/emmanuelmessina00/Scrivania/xai-project5/src/results/feature_extraction/biomedclip_openi_with_reports.pt


AttributeError: 'dict' object has no attribute 'size'

In [ ]:
checkpoint_path = os.path.join(save_dir, 'sae_checkpoint.pt')
start_epoch = 0
history = {'total': [], 'mse': [], 'l1': []}

# Resume from checkpoint if interrupted
if os.path.exists(checkpoint_path):
    print("Found training checkpoint! Resuming...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    sae.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    print(f"Resuming from epoch {start_epoch + 1}...")

print("\n--- STARTING SAE TRAINING ---")
for epoch in range(start_epoch, EPOCHS):
    sae.train()
    epoch_total_loss = 0.0
    epoch_mse_loss = 0.0
    epoch_l1_loss = 0.0

    for batch in dataloader:
        x = batch[0].to(device)
        
        # Forward pass
        x_hat, z = sae(x)
        
        # Calculate loss
        total_loss, mse_loss, l1_loss = sae_loss_function(x, x_hat, z, L1_LAMBDA)
        
        # Backward and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # Normalize decoder weights to prevent collapse (force norm=1)
        sae.normalize_decoder_weights()

        epoch_total_loss += total_loss.item()
        epoch_mse_loss += mse_loss.item()
        epoch_l1_loss += l1_loss.item()
    
    avg_total_loss = epoch_total_loss / len(dataloader)
    avg_mse = epoch_mse_loss / len(dataloader)
    avg_l1 = epoch_l1_loss / len(dataloader)
    
    history['total'].append(avg_total_loss)
    history['mse'].append(avg_mse)
    history['l1'].append(avg_l1)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
          f"Total Loss: {avg_total_loss:.4e} | "
          f"MSE (Reconstruction): {avg_mse:.4e} | "
          f"L1 (Sparsity): {avg_l1:.4e}")
          
    # Save checkpoint at the end of each epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': sae.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history
    }, checkpoint_path)

# Save final model properly for Notebook 04
final_model_path = os.path.join(save_dir, 'sae_model_final.pt')
torch.save(sae.state_dict(), final_model_path)
print(f"\nTraining complete! Final model saved to: {final_model_path}")


## 3. Training Data Analysis
Let's visualize the training metrics to ensure the SAE converged properly and correctly balanced the trade-off between Reconstruction (MSE) and Sparsity (L1).

In [ ]:
epochs_range = range(1, len(history['total']) + 1)

plt.figure(figsize=(14, 5))

# Plot 1: Reconstruction vs Sparsity Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['mse'], label='MSE Loss (Reconstruction)', color='blue', linewidth=2)
plt.plot(epochs_range, history['l1'], label='L1 Loss (Sparsity Penalty)', color='orange', linewidth=2)
plt.title('Loss Components over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.yscale('log') # Log scale for better visibility if differences are huge
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Total Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['total'], label='Total Loss', color='green', linewidth=2)
plt.title('Total Optimization Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()
plt.show()


## 4. Global Explainability: Semantic Grounding

Global explainability aims to map and understand the model's internal architecture in its entirety, independently of individual input samples. In this paradigm, the goal is to build a dictionary that connects the latent visual concepts—learned in a totally unsupervised manner—to human language.

This process, known as *semantic grounding*, is achieved by computing the cosine similarity between the textual embeddings of known clinical concepts (from Notebook 02) and the decoder dictionary learned by the Sparse Autoencoder.

Since both vectors are subjected to L2 normalization, calculating the cosine similarity elegantly reduces to a single matrix multiplication: $S = T \cdot W_{dec}$, where $T$ represents the text embeddings matrix and $W_{dec}$ is the SAE decoder dictionary. This geometric operation allows us to establish the "at-rest" alignment of the network, assigning the most strongly connected latent neuron to each medical phrase.

In [ ]:
dict_path = os.path.join(dict_dir, 'medical_dictionary_embeddings.pt')
print(f"Loading Medical Dictionary from: {dict_path}")

dictionary_data = torch.load(dict_path, map_location=device)
T_matrix = dictionary_data['embeddings'].to(device) # Shape: (N_concepts, 512)
medical_concepts = dictionary_data['concepts'] # List of N strings

print(f"Loaded {len(medical_concepts)} semantic medical concepts.")

# Compute Cosine Similarity
with torch.no_grad():
    sae_dictionary = F.normalize(sae.decoder.weight.data, p=2, dim=0)

similarities = torch.matmul(T_matrix, sae_dictionary)

top_k = 3
# We'll print just a sample (first 5 concepts) to avoid cluttering the output
sample_size = min(5, len(medical_concepts))
print(f"\nShowing Grounding for the first {sample_size} concepts:\n")

for idx in range(sample_size):
    concept = medical_concepts[idx]
    concept_sims = similarities[idx]
    
    # Get top K values and their indices (the SAE "neurons")
    top_values, top_indices = torch.topk(concept_sims, top_k)
    
    print(f"Textual Concept: '{concept}'")
    for i in range(top_k):
        print(f"  -> SAE Neuron {top_indices[i].item():4d} (Similarity: {top_values[i].item():.4f})")
    print("-" * 40)


## 5. Local Explainability: Interpreting a Single Sample

Unlike the global approach, local explainability focuses on interpreting the model's dynamic behavior with respect to a specific input data point. The goal here is to query the neural network to obtain an explanation illustrating which visual concepts were detected within a particular X-ray at the exact moment it is processed.

By providing an image to the Vision-Language Model and projecting its resulting vector into the Sparse Autoencoder, we obtain a latent activation vector $z$. In this context, the network reacts to the visual stimulus by physically activating a very restricted fraction of its neurons. 

By analyzing the intensity of these activations and exploiting the similarity matrix $S$ previously calculated in the global explainability phase, we can map the highly stimulated neurons to their corresponding textual concepts. This mechanism translates mathematical activations into a readable interpretative diagnosis in natural language, highlighting exactly what the model is "looking at" in the single clinical instance.

In [ ]:
def explain_single_image(path, vlm_model, vlm_processor, sae_model, concept_similarities, concept_names, device, top_k_neurons=5):
    vlm_model.eval()
    sae_model.eval()
    
    # Using a dummy image if path doesn't exist just for demonstration purposes
    if not os.path.exists(path):
        print(f"Image not found at {path}. Using a blank dummy image to test logic.")
        import numpy as np
        image = Image.fromarray(np.zeros((300, 300, 3), dtype=np.uint8))
    else:
        image = Image.open(path)

    if image.mode != "RGB":
        image = image.convert("RGB")

    inputs = vlm_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        vision_outputs = vlm_model.get_image_features(**inputs)
        # Extracting the tensor safely (reverted to pooler_output as required by this specific model)
        vision_tensor = vision_outputs.pooler_output
        vision_embeddings = F.normalize(vision_tensor, p=2, dim=1)

        _, z = sae_model(vision_embeddings)

    activations = z[0] # Extracting the activation vector
    top_activation_values, top_neuron_indices = torch.topk(activations, top_k_neurons)

    found_concepts = False
    for val, neuron_idx in zip(top_activation_values, top_neuron_indices):
        if val.item() <= 0.01:
            continue 
            
        found_concepts = True
        print(f"\n[Neuron {neuron_idx.item():4d}] -> Activation Intensity: {val.item():.4f}")
        neuron_concept_scores = concept_similarities[:, neuron_idx]
        
        best_concept_idx = torch.argmax(neuron_concept_scores).item()
        best_concept_score = neuron_concept_scores[best_concept_idx].item()
        
        print(f"  Top Semantic Concept: '{concept_names[best_concept_idx]}'")
        print(f"  Grounding Score: {best_concept_score:.4f}")

    if not found_concepts:
        print("\nNo sufficient strong activations found for this image.")


# We load the VLM processor and model specifically for the local explanation test
print("Loading VLM for Local Explainability test...")
model_id = "flaviagiammarino/pubmed-clip-vit-base-patch32"
vlm_processor = AutoProcessor.from_pretrained(model_id)
vlm_model = AutoModel.from_pretrained(model_id).to(device)

image_test_path = os.path.join(base_dir, 'src', 'images', 'raggi_x.jpg')
print(f"\nAnalyzing image: {image_test_path}")

explain_single_image(
    path=image_test_path, 
    vlm_model=vlm_model, 
    vlm_processor=vlm_processor, 
    sae_model=sae, 
    concept_similarities=similarities, 
    concept_names=medical_concepts, 
    device=device
)
